# Step 1: Spark Environment Setup
## FedHome_Spark Project

**Student:** Md. Raihan Sobhan  
**Course:** Big Data Analytics  
**Date:** 2026-10-07

---

## Overview

This notebook sets up and verifies the Apache Spark environment for the FedHome_Spark project.

### Objectives
1. Initialize Spark session with optimal configuration
2. Test basic Spark operations
3. Verify Spark MLlib availability
4. Benchmark Spark vs. pandas for basic operations

---

## 1. Spark Session Configuration

In [ ]:
from pyspark.sql import SparkSession
from pyspark.ml.feature import StandardScaler, VectorAssembler
from pyspark.ml.clustering import BisectingKMeans
import pyspark

print(f"PySpark Version: {pyspark.__version__}")
print(f"Spark Version: {pyspark.version.SPARK_VERSION}")

In [ ]:
# Create Spark session optimized for MacBook Pro M5 (24GB RAM)
spark = SparkSession.builder \
    .appName("FedHome_Spark") \
    .master("local[*]") \
    .config("spark.driver.memory", "16g") \
    .config("spark.executor.memory", "8g") \
    .config("spark.sql.shuffle.partitions", "10") \
    .config("spark.default.parallelism", "10") \
    .getOrCreate()

print("Spark Session Created Successfully!")
print(f"Spark UI: http://localhost:4040")
print(f"Available Cores: {spark.sparkContext.defaultParallelism}")

---

## 2. Basic Spark Operations Test

In [ ]:
# Test basic DataFrame operations
from pyspark.sql import functions as F

# Create test DataFrame
data = [(i, f"user_{i}", i * 1.5) for i in range(10000)]
df = spark.createDataFrame(data, ["id", "name", "value"])

print(f"DataFrame Count: {df.count()}")
print(f"Schema:")
df.printSchema()

# Test aggregation
df.agg(
    F.avg("value").alias("avg_value"),
    F.max("value").alias("max_value"),
    F.min("value").alias("min_value")
).show()

---

## 3. Spark MLlib Test

In [ ]:
# Test BisectingKMeans clustering
from pyspark.ml.clustering import BisectingKMeans
from pyspark.ml.feature import VectorAssembler

# Create sample feature data
import numpy as np
np.random.seed(42)

sample_data = []
for i in range(1000):
    features = np.random.randn(10).tolist()
    sample_data.append((i, features))

features_df = spark.createDataFrame(sample_data, ["id", "features_raw"])

# Convert to vector format for MLlib
assembler = VectorAssembler(inputCols=["features_raw"], outputCol="features")
features_vector_df = assembler.transform(features_df).select("features")

# Run BisectingKMeans
bkmeans = BisectingKMeans(k=5, maxIter=10)
model = bkmeans.fit(features_vector_df)

print(f"Clustering Complete!")
print(f"Number of clusters: {model.k}")
print(f"Cluster centers computed: {len(model.clusterCenters())}")

---

## 4. Spark vs Pandas Benchmark

In [ ]:
import time
import pandas as pd

# Generate larger dataset for benchmark
n_rows = 100000
print(f"Benchmarking with {n_rows} rows...")

# Pandas benchmark
start = time.time()
pdf = pd.DataFrame({
    'id': range(n_rows),
    'value': np.random.randn(n_rows)
})
pandas_result = pdf.groupby(pdf['id'] % 10)['value'].mean()
pandas_time = time.time() - start
print(f"Pandas Time: {pandas_time:.4f} seconds")

# Spark benchmark
start = time.time()
sdf = spark.createDataFrame(pdf)
spark_result = sdf.groupBy(sdf['id'] % 10).agg(F.avg('value')).collect()
spark_time = time.time() - start
print(f"Spark Time: {spark_time:.4f} seconds")

print(f"\nSpeedup: {pandas_time/spark_time:.2f}x" if spark_time > 0 else "N/A")

---

## 5. Summary

### Environment Verification Results

| Component | Status | Details |
|-----------|--------|---------|
| Java | ✅ | OpenJDK 17 |
| PySpark | ✅ | Version 4.1.2 |
| Spark Session | ✅ | 16GB driver memory |
| DataFrame Ops | ✅ | Working |
| MLlib | ✅ | BisectingKMeans tested |

### Next Steps
1. Proceed to `02_fedmse_spark_fullscale.ipynb` for FedMSE baseline run
2. Use Spark for distributed data preprocessing
3. Implement FedHome clustering with Spark MLlib

In [ ]:
# Cleanup
spark.stop()
print("Spark session stopped.")